# DSE 200: Homework 4

Course repo: https://github.com/DSE200-2026/DSE200
Assignment spec: https://github.com/DSE200-2026/DSE200/blob/main/HW4.md

**Due:** Thursday, November 12, 2026 at 11:59 PM on Gradescope. Submit this notebook with outputs saved.

One new model on data you already know: XGBoost on the Day 3 fraud data. The features, the split and a baseline are built for you in the setup cell. Your job is to fit the model honestly, turn it into dollars, ask it why, and say what more modelling could and could not buy.

## Instructions

* To obtain full marks for a question, both the answer and the code should be correct.
* Please code the solution in the space provided (`# TODO`), and write answers where it says *Replace this text.*
* Every plot gets a title and labeled axes.
* Don't change the setup cell. Everyone works on the same features and the same split, so the numbers can be compared.

In [ ]:
!uv pip install --system -q xgboost scikit-learn

In [ ]:
# ▶ SETUP -- run first. Loads the fraud data, builds the Day 3 features, splits by month, fits a baseline.
import os, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

def find(name):
    """Return a local path for `name`, downloading it from the course repo if needed."""
    for p in [name, "/content/" + name, "data/" + name, "../" + name]:
        if os.path.exists(p):
            return p
    urllib.request.urlretrieve("https://raw.githubusercontent.com/DSE200-2026/DSE200/main/" + name, name)
    return name

FEE = 0.02   # the bank earns 2% of every legit transaction it approves; a missed fraud loses the whole amount

def net_dollars(amount, is_fraud, flagged):
    """What the bank makes over these transactions if it flags `flagged` (0/1 or True/False)."""
    amount, is_fraud, flagged = map(np.asarray, (amount, is_fraud, flagged))
    earned = FEE * amount * ((flagged == 0) & (is_fraud == 0))
    lost = amount * ((flagged == 0) & (is_fraud == 1))
    return round(float((earned - lost).sum()), 2)

# The Day 3 features: each one looks only at the same customer's EARLIER transactions
a = pd.read_csv(find("fraud_training.csv"), parse_dates=["TransactionDate"]).sort_values("TransactionDate")
g = a.groupby("CustomerID")
a["usual_amount"] = g.Amount.transform(lambda s: s.expanding().median().shift(1))
a["amount_ratio"] = (a.Amount / a.usual_amount).fillna(1)
a["away"] = (a.Location != g.Location.transform("first")).astype(int)
a["new_merchant"] = (~a.duplicated(["CustomerID", "MerchantID"])).astype(int)
a["mins_since_prev"] = g.TransactionDate.diff().dt.total_seconds().div(60).fillna(99999)
a["city_jump"] = ((g.Location.shift() != a.Location) & (a.mins_since_prev < 180)).astype(int)
a["hour"] = a.TransactionDate.dt.hour
a["night"] = a.hour.isin([23, 0, 1, 2, 3, 4]).astype(int)
a["tiny"] = (a.Amount < 5).astype(int)
a["just_under_1k"] = a.Amount.between(900, 999.99).astype(int)
a["refund"] = (a.TransactionType == "refund").astype(int)
a = pd.get_dummies(a, columns=["Location"], dtype=int)
features = (["Amount", "refund", "hour"] + [c for c in a.columns if c.startswith("Location_")] +
            ["amount_ratio", "away", "new_merchant", "mins_since_prev", "city_jump", "night", "tiny", "just_under_1k"])

# Split by time, like Day 4's years: fit on Jan-Jun, tune on July, test on Aug-Sep
fit  = a[a.TransactionDate < "2024-07-01"]
tune = a[(a.TransactionDate >= "2024-07-01") & (a.TransactionDate < "2024-08-01")]
test = a[a.TransactionDate >= "2024-08-01"]

# The baseline to beat: Day 3's logistic regression
baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(fit[features], fit.IsFraud)
p_base = baseline.predict_proba(test[features])[:, 1]

print(f"fit {len(fit):,}   tune {len(tune):,}   test {len(test):,}   fraud share {test.IsFraud.mean():.2%}")
print("baseline PR-AUC on Aug-Sep:", round(average_precision_score(test.IsFraud, p_base), 3))

---
# Part 1: Fit XGBoost (30 pts)

1. Fit `XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=4, early_stopping_rounds=50, eval_metric="aucpr", random_state=42)` on `fit`, with `eval_set=[(tune[features], tune.IsFraud)]`. - **15 pts**
2. Print `best_iteration`, and the PR-AUC of its probabilities on `test` next to the baseline's. - **10 pts**
3. In one sentence: why does early stopping watch July, and not the August-September test months? - **5 pts**

In [ ]:
model = None    # TODO: the XGBClassifier above, fitted with early stopping on July
p_xgb = None    # TODO: model's probability of fraud for every test row

In [ ]:
# TODO: print model.best_iteration, then PR-AUC on test for the baseline (p_base) and for XGBoost (p_xgb)

*Replace this text.*

---
# Part 2: Turn it into dollars (30 pts)

Day 3's cost table: a missed fraud loses the whole amount, a false alarm loses the 2% fee. So flag a transaction when P(fraud) > 0.02 / 1.02.

1. Build one table with **net dollars on the test months** for: approve everything, the baseline at the cost threshold, XGBoost at 0.5, XGBoost at the cost threshold. - **20 pts**
2. In one or two sentences: why does XGBoost at 0.5 do so badly, when its PR-AUC is the best in the table? - **10 pts**

In [ ]:
THRESHOLD = FEE / (1 + FEE)

table = None    # TODO: a pd.Series or DataFrame with the four net-dollar numbers, one per strategy
table

*Replace this text.*

---
# Part 3: Ask it why (20 pts)

1. Permutation importance of `model` on the **test** rows, scored by PR-AUC: `permutation_importance(model, test[features], test.IsFraud, scoring="average_precision", n_repeats=5, random_state=0)`. Plot the top 8 as a horizontal bar chart. - **12 pts**
2. In two sentences: do the top features make sense for fraud? Is any one implausibly important, the way a leak would be? - **8 pts**

In [ ]:
from sklearn.inspection import permutation_importance

# TODO: permutation importance as above, then a horizontal bar chart of the top 8 with a title

*Replace this text.*

---
# Part 4: What more modelling can and cannot buy (20 pts)

Answer in markdown, two to four sentences each.

1. **The ceiling (10 pts).** A colleague wants to run AutoGluon for an hour on the same features and expects a big jump in PR-AUC. What do you expect, and why? Use the idea of the Bayes optimal classifier from Day 4.
2. **Seeing is not doing (10 pts).** `mins_since_prev` is near the top of your chart. A manager proposes making every customer wait 30 minutes between purchases to cut fraud. What did your model actually show, and what would you need to know before doing that?

*Replace this text.*

---
# Bonus (5 pts)

Run **AutoGluon** (with `tuning_data=tune`) or **TabPFN** (on a sample of `fit`, as in Day 4 section 9.3) on the same features, and add its PR-AUC and net dollars at the cost threshold to your Part 2 table. One sentence on what you found.

In [ ]:
# Bonus (optional)